In [9]:
from __future__ import annotations

import json
import subprocess
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable


In [14]:
@dataclass
class EvidenceRecord:
    command: str
    return_code: int
    elapsed_seconds: float
    stdout: str
    stderr: str
    observed_at_utc: str


def run_command_with_evidence(command: str, timeout_seconds: int = 300) -> EvidenceRecord:
    start = time.perf_counter()
    proc = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
    )
    elapsed = time.perf_counter() - start

    return EvidenceRecord(
        command=command,
        return_code=proc.returncode,
        elapsed_seconds=round(elapsed, 4),
        stdout=proc.stdout,
        stderr=proc.stderr,
        observed_at_utc=datetime.now(timezone.utc).isoformat(),
    )


def observe_test_suite(commands: Iterable[str], timeout_seconds: int = 300) -> list[EvidenceRecord]:
    records: list[EvidenceRecord] = []
    for command in commands:
        print(f"Running: {command}")
        record = run_command_with_evidence(command, timeout_seconds=timeout_seconds)
        records.append(record)
        print(f" -> exit={record.return_code}, elapsed={record.elapsed_seconds}s")
    return records


def save_evidence(records: list[EvidenceRecord], output_path: str = "test-evidence.json") -> Path:
    out = Path(output_path)
    payload = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": [asdict(r) for r in records],
    }
    out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Evidence written to: {out.resolve()}")
    return out


Running: python -m unittest discover -v
 -> exit=0, elapsed=0.1066s
Evidence written to: /Users/owner/AI-History/test-evidence.json


[EvidenceRecord(command='python -m unittest discover -v', return_code=0, elapsed_seconds=0.1066, stdout='', stderr='\n----------------------------------------------------------------------\nRan 0 tests in 0.000s\n\nOK\n', observed_at_utc='2026-03-02T11:44:21.797841+00:00')]

In [17]:
# Debug session: improve and validate input before running tests.
import re


In [ ]:

# Paste one command per line.
# Lines starting with # are ignored.
RAW_TEST_INPUT = """
python -m unittest discover -v
"""


def parse_test_commands(raw_text: str) -> list[str]:
    commands: list[str] = []
    for line in raw_text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        commands.append(stripped)
    return commands


def debug_input_quality(commands: list[str]) -> dict:
    warnings: list[str] = []

    if not commands:
        warnings.append("No commands provided. Add at least one test command.")

    suspicious_patterns = [r"\brm\b", r"\bshutdown\b", r"\breboot\b", r"git\s+reset\s+--hard"]
    for cmd in commands:
        for pattern in suspicious_patterns:
            if re.search(pattern, cmd):
                warnings.append(f"Potentially unsafe command detected: {cmd}")
                break

        if "test" not in cmd and "unittest" not in cmd and "pytest" not in cmd:
            warnings.append(f"Command may not be a test command: {cmd}")

    return {
        "command_count": len(commands),
        "commands": commands,
        "warnings": warnings,
    }


TEST_COMMANDS = parse_test_commands(RAW_TEST_INPUT)
input_report = debug_input_quality(TEST_COMMANDS)

if input_report["warnings"]:
    print("Input warnings detected:")
    for warning in input_report["warnings"]:
        print(f" - {warning}")

records = observe_test_suite(TEST_COMMANDS, timeout_seconds=300)
evidence_file = save_evidence(records, output_path="test-evidence.json")

input_report, records


In [16]:
# Basic unit-test style assertions over observed evidence.
# Set this to True once the repository contains real tests.
REQUIRE_AT_LEAST_ONE_TEST = False

for record in records:
    no_tests_ran = (
        "NO TESTS RAN" in record.stdout
        or "Ran 0 tests" in record.stdout
    )

    if REQUIRE_AT_LEAST_ONE_TEST:
        assert not no_tests_ran, (
            f"No tests executed for command: {record.command}\n"
            f"STDOUT:\n{record.stdout}"
        )

    assert record.return_code in (0, 5), (
        f"Unexpected failure: {record.command}\n"
        f"STDOUT:\n{record.stdout}\n"
        f"STDERR:\n{record.stderr}"
    )

print("Evidence checks completed. Set REQUIRE_AT_LEAST_ONE_TEST=True when tests are added.")


Evidence checks completed. Set REQUIRE_AT_LEAST_ONE_TEST=True when tests are added.


In [ ]:
# End of debug session.
# Add any ad-hoc checks below as needed.